<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-09/AI_Modul_9.2_Praktikum_Artificial_Neural_Network.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 9.2: Praktikum Artificial Neural Network (ANN)
**Mata Kuliah:** Kecerdasan Buatan Terapan & Deep Learning  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Membangun kelas arsitektur Multi-Layer Perceptron (MLP) tervektorisasi menggunakan NumPy murni.
2. Mengimplementasikan inisialisasi bobot He (Kaiming) dan propagasi maju terstruktur.
3. Menguji aktivasi Softmax dengan stabilisasi numerik pada data evaluasi kesesuaian lahan kelapa sawit.
4. Memvisualisasikan batas keputusan dan menganalisis representasi ruang laten.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'NumPy Version: {np.__version__}')


## 1. Pembangkitan Data Sintetis Sifat Tanah dan Kesesuaian Lahan
Mensimulasikan 400 sampel petak tanah perkebunan dengan dua fitur utama: Derajat Keasaman Tanah (pH) dan Kapasitas Tukar Kation (KTK).

In [ ]:
np.random.seed(42)
n_per_class = 120

# Kelas 0: S1 (Sangat Sesuai)
c0 = np.random.multivariate_normal([5.8, 22.0], [[0.15, 0.1], [0.1, 3.0]], n_per_class)
# Kelas 1: S2 (Cukup Sesuai)
c1 = np.random.multivariate_normal([5.0, 15.0], [[0.2, -0.05], [-0.05, 2.5]], n_per_class)
# Kelas 2: S3/N (Marginal/Tidak Sesuai)
c2 = np.random.multivariate_normal([4.3, 9.0], [[0.15, 0.05], [0.05, 2.0]], n_per_class)

X_raw = np.vstack([c0, c1, c2])
y_true = np.array([0]*n_per_class + [1]*n_per_class + [2]*n_per_class)

# Standarisasi Fitur
X_std = (X_raw - np.mean(X_raw, axis=0)) / np.std(X_raw, axis=0)

print('Bentuk Matriks Fitur Tanah:', X_std.shape)
print('Distribusi Label Target   :', np.bincount(y_true))


## 2. Implementasi Kelas Multi-Layer Perceptron (Pure NumPy)
Membangun kelas jaringan saraf dengan konfigurasi lapisan fleksibel, aktivasi ReLU tersembunyi, dan Softmax keluaran.

In [ ]:
class MultiLayerPerceptron:
    def __init__(self, layer_dims, seed=42):
        np.random.seed(seed)
        self.layer_dims = layer_dims
        self.num_layers = len(layer_dims) - 1
        self.parameters = {}
        
        for l in range(1, len(layer_dims)):
            # Inisialisasi He (Kaiming)
            self.parameters[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2.0 / layer_dims[l-1])
            self.parameters[f'b{l}'] = np.zeros((1, layer_dims[l]))
            
    def relu(self, Z):
        return np.maximum(0, Z)
        
    def softmax(self, Z):
        exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
        
    def forward(self, X):
        A = X
        cache = {'A0': X}
        for l in range(1, self.num_layers):
            W = self.parameters[f'W{l}']
            b = self.parameters[f'b{l}']
            Z = np.dot(A, W.T) + b
            A = self.relu(Z)
            cache[f'Z{l}'] = Z
            cache[f'A{l}'] = A
            
        # Output layer
        W_out = self.parameters[f'W{self.num_layers}']
        b_out = self.parameters[f'b{self.num_layers}']
        Z_out = np.dot(A, W_out.T) + b_out
        A_out = self.softmax(Z_out)
        cache[f'Z{self.num_layers}'] = Z_out
        cache[f'A{self.num_layers}'] = A_out
        return A_out, cache

model = MultiLayerPerceptron(layer_dims=[2, 8, 4, 3])
probs, cache = model.forward(X_std)
preds = np.argmax(probs, axis=1)
print(f'Forward pass berhasil dieksekusi! Output shape: {probs.shape}')
print('Contoh 3 estimasi probabilitas pertama:\n', probs[:3])


## 3. Visualisasi Sebaran Fitur Tanah dan Ruang Keputusan
Memetakan sebaran sampel tanah dan ruang klasifikasi kesesuaian lahan.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_std[:, 0], X_std[:, 1], c=y_true, cmap='viridis', edgecolors='k', s=60, alpha=0.85)
ax.set_title('Distribusi Karakteristik Tanah Perkebunan (pH vs KTK)', fontsize=12, fontweight='bold')
ax.set_xlabel('Derajat Keasaman Tanah (pH Terstandarisasi)', fontsize=10)
ax.set_ylabel('Kapasitas Tukar Kation (KTK Terstandarisasi)', fontsize=10)
legend1 = ax.legend(*scatter.legend_elements(), title='Kelas Lahan', loc='upper left')
ax.add_artist(legend1)
plt.tight_layout()
plt.savefig('praktikum_sebaran_tanah_9_2.png', dpi=150)
plt.show()
